# Module 4.1: 语言建模基础

## 1. 本章概览

### 📚 学习目标

1. **语言模型基础**：理解语言建模的核心概念和目标
2. **统计语言模型**：掌握N-gram模型及其平滑技术
3. **神经语言模型**：从RNN到Transformer的演进
4. **评估指标**：理解困惑度(Perplexity)等评估方法
5. **文本生成**：掌握各种解码策略

### 🎯 核心问题

- 什么是语言建模？为什么重要？
- N-gram模型的优缺点是什么？
- 神经语言模型如何解决稀疏性问题？
- 如何评估语言模型的好坏？
- 不同的文本生成策略有什么区别？

### 🗺️ 知识地图

```
统计语言模型
├── Unigram (词频)
├── Bigram (二元语法)
└── N-gram (多元语法)
    ↓
神经语言模型
├── RNN-LM
├── LSTM-LM
└── Transformer-LM (GPT)
    ↓
预训练语言模型
└── BERT, GPT-3, etc.
```

### ⏱️ 预计学习时间：3-4小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import re

np.random.seed(42)
torch.manual_seed(42)
print("✓ Libraries imported")

## 2. 动机与背景

### 什么是语言建模？

**语言模型(Language Model, LM)**：对自然语言序列的概率分布进行建模。

**核心任务**：给定前面的词，预测下一个词的概率。

**数学表示**：

对于句子 $w_1, w_2, ..., w_n$，语言模型计算其概率：

$$P(w_1, w_2, ..., w_n) = \prod_{i=1}^n P(w_i | w_1, ..., w_{i-1})$$

**应用场景**：
- 📝 **文本生成**：写作助手、对话系统
- 🔍 **语音识别**：选择最可能的句子
- 🌐 **机器翻译**：评估翻译质量
- ✍️ **拼写纠错**：判断哪个词更合理
- 🤖 **预训练模型**：BERT、GPT的基础

### 为什么重要？

语言建模是NLP的基础任务：
1. **理解语言结构**：学习语法和语义规律
2. **表示学习**：学习词和句子的分布式表示
3. **迁移学习**：预训练模型的核心任务

## 3. 理论基础

### 3.1 统计语言模型

**Unigram模型**：最简单的语言模型，假设词之间独立。

$$P(w_1, w_2, ..., w_n) = \prod_{i=1}^n P(w_i)$$

其中 $P(w_i) = \frac{\text{count}(w_i)}{\text{total words}}$

**优点**：简单、快速
**缺点**：忽略上下文，生成的文本不连贯

In [ ]:
# 🔬 Micro Practice: Implement Unigram Language Model
# Goal: Build simplest language model
# Expected outcome: Understand word frequency-based modeling

class UnigramLM:
    """
    Unigram Language Model
    """
    
    def __init__(self):
        self.word_counts = Counter()
        self.total_words = 0
        self.vocab = set()
    
    def train(self, corpus):
        """
        Train on corpus
        
        Args:
            corpus: List of sentences (each sentence is a list of words)
        """
        for sentence in corpus:
            for word in sentence:
                self.word_counts[word] += 1
                self.vocab.add(word)
        
        self.total_words = sum(self.word_counts.values())
    
    def probability(self, word):
        """
        Get probability of a word
        
        Args:
            word: Word to get probability for
        
        Returns:
            Probability P(word)
        """
        return self.word_counts[word] / self.total_words if self.total_words > 0 else 0
    
    def sentence_probability(self, sentence):
        """
        Get probability of a sentence (product of word probabilities)
        
        Args:
            sentence: List of words
        
        Returns:
            Probability P(sentence)
        """
        prob = 1.0
        for word in sentence:
            prob *= self.probability(word)
        return prob
    
    def generate(self, length=10):
        """
        Generate text by sampling from word distribution
        
        Args:
            length: Number of words to generate
        
        Returns:
            Generated sentence
        """
        words = list(self.word_counts.keys())
        probs = [self.probability(w) for w in words]
        
        generated = np.random.choice(words, size=length, p=probs)
        return ' '.join(generated)

# Example corpus
corpus = [
    ['the', 'cat', 'sat', 'on', 'the', 'mat'],
    ['the', 'dog', 'sat', 'on', 'the', 'log'],
    ['cats', 'and', 'dogs', 'are', 'animals'],
    ['the', 'cat', 'and', 'the', 'dog']
]

# Train unigram model
unigram = UnigramLM()
unigram.train(corpus)

print("=== Unigram Language Model ===\n")
print(f"Vocabulary size: {len(unigram.vocab)}")
print(f"Total words: {unigram.total_words}\n")

# Word probabilities
print("Top 5 most frequent words:")
for word, count in unigram.word_counts.most_common(5):
    prob = unigram.probability(word)
    print(f"  {word}: count={count}, P(w)={prob:.3f}")
print()

# Sentence probability
test_sentence = ['the', 'cat', 'sat']
prob = unigram.sentence_probability(test_sentence)
print(f"P({' '.join(test_sentence)}) = {prob:.6f}\n")

# Generate text
print("Generated text (3 samples):")
for i in range(3):
    print(f"  {i+1}. {unigram.generate(8)}")

print("\n观察：生成的文本词频合理，但缺乏连贯性")
print("✓ Unigram模型实现完成！")

### 3.2 N-gram语言模型

**马尔可夫假设**：当前词只依赖于前面有限个词。

**N-gram模型**：

- **Bigram (2-gram)**：$P(w_i | w_{i-1})$
- **Trigram (3-gram)**：$P(w_i | w_{i-2}, w_{i-1})$
- **N-gram**：$P(w_i | w_{i-n+1}, ..., w_{i-1})$

**条件概率**：

$$P(w_i | w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i)}{\text{count}(w_{i-1})}$$

**稀疏性问题**：

许多n-gram在训练数据中从未出现，导致概率为0。

**平滑 (Smoothing)** 是一类用于解决 N-gram 模型中零概率问题的技术，通过重新分配概率质量，使未见过的 N-gram 也能获得非零概率。在本章场景中，它用于缓解训练数据稀疏导致的语言模型概率估计偏差。

**常见平滑方法**：

1. **Add-k平滑**：给所有计数加上k
   $$P(w_i | w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i) + k}{\text{count}(w_{i-1}) + k|V|}$$

2. **回退(Backoff)**：如果n-gram未见过，使用(n-1)-gram

3. **插值(Interpolation)**：混合不同阶的n-gram

**权衡**：
- N越大：更多上下文，但更稀疏
- N越小：更少稀疏，但上下文有限

In [ ]:
# 🔬 Micro Practice: Implement Bigram Language Model
# Goal: Build bigram model with smoothing
# Expected outcome: Understand conditional probability and smoothing

class BigramLM:
    """
    Bigram Language Model with Add-k Smoothing
    """
    
    def __init__(self, k=1.0):
        """
        Args:
            k: Smoothing parameter (default: 1.0 for add-1 smoothing)
        """
        self.k = k
        self.bigram_counts = defaultdict(Counter)  # {w1: {w2: count}}
        self.unigram_counts = Counter()
        self.vocab = set()
    
    def train(self, corpus):
        """
        Train on corpus
        
        Args:
            corpus: List of sentences (each sentence is a list of words)
        """
        for sentence in corpus:
            # Add start and end tokens
            sentence = ['<START>'] + sentence + ['<END>']
            
            for i in range(len(sentence) - 1):
                w1, w2 = sentence[i], sentence[i+1]
                self.bigram_counts[w1][w2] += 1
                self.unigram_counts[w1] += 1
                self.vocab.add(w1)
                self.vocab.add(w2)
    
    def probability(self, w1, w2):
        """
        Get conditional probability P(w2 | w1) with add-k smoothing
        
        Args:
            w1: Previous word
            w2: Current word
        
        Returns:
            P(w2 | w1)
        """
        numerator = self.bigram_counts[w1][w2] + self.k
        denominator = self.unigram_counts[w1] + self.k * len(self.vocab)
        return numerator / denominator if denominator > 0 else 0
    
    def sentence_probability(self, sentence):
        """
        Get probability of a sentence
        
        Args:
            sentence: List of words
        
        Returns:
            P(sentence)
        """
        sentence = ['<START>'] + sentence + ['<END>']
        prob = 1.0
        
        for i in range(len(sentence) - 1):
            prob *= self.probability(sentence[i], sentence[i+1])
        
        return prob
    
    def generate(self, max_length=20):
        """
        Generate text using bigram model
        
        Args:
            max_length: Maximum number of words to generate
        
        Returns:
            Generated sentence
        """
        current = '<START>'
        generated = []
        
        for _ in range(max_length):
            # Get possible next words
            next_words = list(self.bigram_counts[current].keys())
            if not next_words or current == '<END>':
                break
            
            # Calculate probabilities
            probs = [self.probability(current, w) for w in next_words]
            probs = np.array(probs)
            probs = probs / probs.sum()  # Normalize
            
            # Sample next word
            current = np.random.choice(next_words, p=probs)
            
            if current == '<END>':
                break
            
            generated.append(current)
        
        return ' '.join(generated)

# Train bigram model
bigram = BigramLM(k=1.0)
bigram.train(corpus)

print("=== Bigram Language Model ===\n")
print(f"Vocabulary size: {len(bigram.vocab)}")
print(f"Number of bigrams: {sum(len(v) for v in bigram.bigram_counts.values())}\n")

# Conditional probabilities
print("Conditional probabilities:")
print(f"  P('cat' | 'the') = {bigram.probability('the', 'cat'):.3f}")
print(f"  P('dog' | 'the') = {bigram.probability('the', 'dog'):.3f}")
print(f"  P('sat' | 'cat') = {bigram.probability('cat', 'sat'):.3f}\n")

# Sentence probability
test_sentence = ['the', 'cat', 'sat']
prob = bigram.sentence_probability(test_sentence)
print(f"P({' '.join(test_sentence)}) = {prob:.6f}\n")

# Generate text
print("Generated text (3 samples):")
for i in range(3):
    print(f"  {i+1}. {bigram.generate(10)}")

print("\n观察：生成的文本开始有一些连贯性")
print("✓ Bigram模型实现完成！")

In [ ]:
# 🔬 Micro Practice: Implement Trigram Language Model
# Goal: Build trigram model and compare with bigram
# Expected outcome: Understand trade-off between context and sparsity

class TrigramLM:
    """
    Trigram Language Model with Add-k Smoothing
    """
    
    def __init__(self, k=1.0):
        """
        Args:
            k: Smoothing parameter
        """
        self.k = k
        self.trigram_counts = defaultdict(lambda: defaultdict(Counter))  # {w1: {w2: {w3: count}}}
        self.bigram_counts = defaultdict(Counter)
        self.vocab = set()
    
    def train(self, corpus):
        """
        Train on corpus
        
        Args:
            corpus: List of sentences (each sentence is a list of words)
        """
        for sentence in corpus:
            # Add start and end tokens
            sentence = ['<START>'] + sentence + ['<END>']
            
            for i in range(len(sentence) - 2):
                w1, w2, w3 = sentence[i], sentence[i+1], sentence[i+2]
                self.trigram_counts[w1][w2][w3] += 1
                self.bigram_counts[(w1, w2)][w3] += 1
                self.vocab.add(w1)
                self.vocab.add(w2)
                self.vocab.add(w3)
    
    def probability(self, w1, w2, w3):
        """
        Get conditional probability P(w3 | w1, w2) with add-k smoothing
        
        Args:
            w1: Two words before
            w2: Previous word
            w3: Current word
        
        Returns:
            P(w3 | w1, w2)
        """
        numerator = self.trigram_counts[w1][w2][w3] + self.k
        denominator = sum(self.trigram_counts[w1][w2].values()) + self.k * len(self.vocab)
        return numerator / denominator if denominator > 0 else 1.0 / len(self.vocab)
    
    def sentence_probability(self, sentence):
        """
        Get probability of a sentence
        
        Args:
            sentence: List of words
        
        Returns:
            P(sentence)
        """
        sentence = ['<START>'] + sentence + ['<END>']
        prob = 1.0
        
        for i in range(len(sentence) - 2):
            prob *= self.probability(sentence[i], sentence[i+1], sentence[i+2])
        
        return prob
    
    def generate(self, max_length=20):
        """
        Generate text using trigram model
        
        Args:
            max_length: Maximum number of words to generate
        
        Returns:
            Generated sentence
        """
        w1, w2 = '<START>', '<START>'
        generated = []
        
        for _ in range(max_length):
            # Get possible next words
            if w1 in self.trigram_counts and w2 in self.trigram_counts[w1]:
                next_words = list(self.trigram_counts[w1][w2].keys())
            else:
                break
            
            if not next_words:
                break
            
            # Calculate probabilities
            probs = [self.probability(w1, w2, w) for w in next_words]
            probs = np.array(probs)
            probs = probs / probs.sum()  # Normalize
            
            # Sample next word
            w3 = np.random.choice(next_words, p=probs)
            
            if w3 == '<END>':
                break
            
            generated.append(w3)
            w1, w2 = w2, w3
        
        return ' '.join(generated)

# Train trigram model
trigram = TrigramLM(k=1.0)
trigram.train(corpus)

print("=== Trigram Language Model ===\n")
print(f"Vocabulary size: {len(trigram.vocab)}")
print(f"Number of trigrams: {sum(sum(len(v2) for v2 in v1.values()) for v1 in trigram.trigram_counts.values())}\n")

# Conditional probabilities
print("Conditional probabilities:")
print(f"  P('sat' | 'the', 'cat') = {trigram.probability('the', 'cat', 'sat'):.3f}")
print(f"  P('sat' | 'the', 'dog') = {trigram.probability('the', 'dog', 'sat'):.3f}\n")

# Sentence probability
test_sentence = ['the', 'cat', 'sat']
prob = trigram.sentence_probability(test_sentence)
print(f"P({' '.join(test_sentence)}) = {prob:.6f}\n")

# Generate text
print("Generated text (3 samples):")
for i in range(3):
    print(f"  {i+1}. {trigram.generate(10)}")

print("\n观察：生成的文本更加连贯，但可能更接近训练数据")
print("✓ Trigram模型实现完成！")

In [ ]:
# 🔬 Micro Practice: Compare N-gram Models
# Goal: Visualize trade-off between context and sparsity
# Expected outcome: Understand when to use which model

# Compare models on same test sentence
test_sentences = [
    ['the', 'cat', 'sat'],
    ['the', 'dog', 'sat'],
    ['cats', 'and', 'dogs']
]

print("=== Model Comparison ===\n")
print("Sentence Probabilities:\n")
print(f"{'Sentence':<20} {'Unigram':<15} {'Bigram':<15} {'Trigram':<15}")
print("-" * 65)

for sentence in test_sentences:
    sent_str = ' '.join(sentence)
    p_uni = unigram.sentence_probability(sentence)
    p_bi = bigram.sentence_probability(sentence)
    p_tri = trigram.sentence_probability(sentence)
    print(f"{sent_str:<20} {p_uni:<15.8f} {p_bi:<15.8f} {p_tri:<15.8f}")

print("\n" + "="*65 + "\n")

# Visualize model statistics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Model sizes
models = ['Unigram', 'Bigram', 'Trigram']
vocab_sizes = [len(unigram.vocab), len(bigram.vocab), len(trigram.vocab)]
n_grams = [
    len(unigram.word_counts),
    sum(len(v) for v in bigram.bigram_counts.values()),
    sum(sum(len(v2) for v2 in v1.values()) for v1 in trigram.trigram_counts.values())
]

axes[0].bar(models, vocab_sizes, color=['blue', 'green', 'orange'])
axes[0].set_title('Vocabulary Size', fontsize=12, weight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(models, n_grams, color=['blue', 'green', 'orange'])
axes[1].set_title('Number of N-grams', fontsize=12, weight='bold')
axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.3, axis='y')

# Generation quality (measured by average sentence length)
gen_lengths = []
for model in [unigram, bigram, trigram]:
    lengths = []
    for _ in range(10):
        if hasattr(model, 'generate'):
            if model == unigram:
                text = model.generate(8)
            else:
                text = model.generate(20)
            lengths.append(len(text.split()))
    gen_lengths.append(np.mean(lengths))

axes[2].bar(models, gen_lengths, color=['blue', 'green', 'orange'])
axes[2].set_title('Avg Generated Length', fontsize=12, weight='bold')
axes[2].set_ylabel('Words')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("观察：")
print("- N越大：更多上下文信息，生成更连贯")
print("- N越大：n-gram数量增加，稀疏性问题加剧")
print("- 实践中：通常使用trigram或4-gram + 平滑技术")
print("\n✓ N-gram模型比较完成！")

## 4. 评估指标

### 4.1 困惑度 (Perplexity)

**困惑度 (Perplexity, PPL)** 是衡量语言模型预测能力的核心评估指标，数值越低表示模型对测试数据的预测越准确。在本章场景中，它用于比较不同语言模型（N-gram、RNN、Transformer）的建模质量。

**数学表示**：

对于测试集 $w_1, w_2, ..., w_N$：

$$\text{Perplexity} = P(w_1, w_2, ..., w_N)^{-\frac{1}{N}}$$

展开为：

$$\text{PPL} = \sqrt[N]{\frac{1}{P(w_1, w_2, ..., w_N)}}$$

使用对数形式（避免数值下溢）：

$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^N \log P(w_i | w_1, ..., w_{i-1})\right)$$

**直观理解**：

- 困惑度表示模型在每个位置"困惑"于多少个词
- PPL = 10：模型平均在10个词中犹豫
- PPL = 100：模型平均在100个词中犹豫
- **越低越好**：PPL = 1 表示完美预测

**与交叉熵的关系**：

$$\text{PPL} = 2^{H(p, q)} = \exp(H(p, q))$$

其中 $H(p, q)$ 是交叉熵。

**为什么使用困惑度？**

1. **可解释性**：比交叉熵更直观
2. **标准化**：不受序列长度影响
3. **广泛使用**：语言模型的标准评估指标

In [ ]:
# 🔬 Micro Practice: Implement Perplexity Calculation
# Goal: Build perplexity metric for language models
# Expected outcome: Understand how to evaluate LMs

def calculate_perplexity(model, test_corpus, model_type='unigram'):
    """
    Calculate perplexity of a language model on test corpus
    
    Args:
        model: Language model (unigram, bigram, or trigram)
        test_corpus: List of test sentences
        model_type: Type of model ('unigram', 'bigram', 'trigram')
    
    Returns:
        Perplexity value
    """
    log_prob_sum = 0
    word_count = 0
    
    for sentence in test_corpus:
        if model_type == 'unigram':
            # For unigram: P(w1, w2, ..., wn) = P(w1) * P(w2) * ... * P(wn)
            for word in sentence:
                prob = model.probability(word)
                if prob > 0:
                    log_prob_sum += np.log(prob)
                else:
                    log_prob_sum += np.log(1e-10)  # Avoid log(0)
                word_count += 1
        
        elif model_type == 'bigram':
            # For bigram: use conditional probabilities
            sentence_with_tokens = ['<START>'] + sentence + ['<END>']
            for i in range(len(sentence_with_tokens) - 1):
                prob = model.probability(sentence_with_tokens[i], sentence_with_tokens[i+1])
                if prob > 0:
                    log_prob_sum += np.log(prob)
                else:
                    log_prob_sum += np.log(1e-10)
                word_count += 1
        
        elif model_type == 'trigram':
            # For trigram: use conditional probabilities
            sentence_with_tokens = ['<START>'] + sentence + ['<END>']
            for i in range(len(sentence_with_tokens) - 2):
                prob = model.probability(
                    sentence_with_tokens[i], 
                    sentence_with_tokens[i+1], 
                    sentence_with_tokens[i+2]
                )
                if prob > 0:
                    log_prob_sum += np.log(prob)
                else:
                    log_prob_sum += np.log(1e-10)
                word_count += 1
    
    # Calculate perplexity
    avg_log_prob = log_prob_sum / word_count
    perplexity = np.exp(-avg_log_prob)
    
    return perplexity

# Create test corpus (different from training)
test_corpus = [
    ['the', 'cat', 'and', 'dog'],
    ['cats', 'are', 'animals'],
    ['the', 'dog', 'sat']
]

print("=== Perplexity Evaluation ===\n")
print("Test corpus:")
for i, sent in enumerate(test_corpus):
    print(f"  {i+1}. {' '.join(sent)}")
print()

# Calculate perplexity for each model
ppl_unigram = calculate_perplexity(unigram, test_corpus, 'unigram')
ppl_bigram = calculate_perplexity(bigram, test_corpus, 'bigram')
ppl_trigram = calculate_perplexity(trigram, test_corpus, 'trigram')

print("Perplexity Results:")
print(f"  Unigram:  {ppl_unigram:.2f}")
print(f"  Bigram:   {ppl_bigram:.2f}")
print(f"  Trigram:  {ppl_trigram:.2f}")
print()

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

models = ['Unigram', 'Bigram', 'Trigram']
perplexities = [ppl_unigram, ppl_bigram, ppl_trigram]
colors = ['blue', 'green', 'orange']

bars = ax.bar(models, perplexities, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add value labels on bars
for bar, ppl in zip(bars, perplexities):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{ppl:.2f}',
            ha='center', va='bottom', fontsize=12, weight='bold')

ax.set_ylabel('Perplexity (lower is better)', fontsize=12, weight='bold')
ax.set_title('Language Model Comparison on Test Set', fontsize=14, weight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(perplexities) * 1.2)

plt.tight_layout()
plt.show()

print("观察：")
print("- 困惑度越低，模型越好")
print("- Bigram/Trigram通常比Unigram表现更好")
print("- 但在小数据集上，可能出现过拟合")
print("\n✓ 困惑度计算完成！")

In [ ]:
# 🔬 Micro Practice: Other Evaluation Metrics
# Goal: Implement cross-entropy and bits-per-character
# Expected outcome: Understand different evaluation perspectives

def calculate_cross_entropy(model, test_corpus, model_type='unigram'):
    """
    Calculate cross-entropy (average negative log-likelihood)
    
    Args:
        model: Language model
        test_corpus: List of test sentences
        model_type: Type of model
    
    Returns:
        Cross-entropy value
    """
    log_prob_sum = 0
    word_count = 0
    
    for sentence in test_corpus:
        if model_type == 'unigram':
            for word in sentence:
                prob = model.probability(word)
                log_prob_sum += np.log(prob) if prob > 0 else np.log(1e-10)
                word_count += 1
        
        elif model_type == 'bigram':
            sentence_with_tokens = ['<START>'] + sentence + ['<END>']
            for i in range(len(sentence_with_tokens) - 1):
                prob = model.probability(sentence_with_tokens[i], sentence_with_tokens[i+1])
                log_prob_sum += np.log(prob) if prob > 0 else np.log(1e-10)
                word_count += 1
        
        elif model_type == 'trigram':
            sentence_with_tokens = ['<START>'] + sentence + ['<END>']
            for i in range(len(sentence_with_tokens) - 2):
                prob = model.probability(
                    sentence_with_tokens[i], 
                    sentence_with_tokens[i+1], 
                    sentence_with_tokens[i+2]
                )
                log_prob_sum += np.log(prob) if prob > 0 else np.log(1e-10)
                word_count += 1
    
    cross_entropy = -log_prob_sum / word_count
    return cross_entropy

def calculate_bits_per_char(model, test_corpus, model_type='unigram'):
    """
    Calculate bits per character
    
    Args:
        model: Language model
        test_corpus: List of test sentences
        model_type: Type of model
    
    Returns:
        Bits per character
    """
    log_prob_sum = 0
    char_count = 0
    
    for sentence in test_corpus:
        # Count characters (including spaces)
        char_count += sum(len(word) for word in sentence) + len(sentence) - 1
        
        if model_type == 'unigram':
            for word in sentence:
                prob = model.probability(word)
                log_prob_sum += np.log2(prob) if prob > 0 else np.log2(1e-10)
        
        elif model_type == 'bigram':
            sentence_with_tokens = ['<START>'] + sentence + ['<END>']
            for i in range(len(sentence_with_tokens) - 1):
                prob = model.probability(sentence_with_tokens[i], sentence_with_tokens[i+1])
                log_prob_sum += np.log2(prob) if prob > 0 else np.log2(1e-10)
        
        elif model_type == 'trigram':
            sentence_with_tokens = ['<START>'] + sentence + ['<END>']
            for i in range(len(sentence_with_tokens) - 2):
                prob = model.probability(
                    sentence_with_tokens[i], 
                    sentence_with_tokens[i+1], 
                    sentence_with_tokens[i+2]
                )
                log_prob_sum += np.log2(prob) if prob > 0 else np.log2(1e-10)
    
    bpc = -log_prob_sum / char_count
    return bpc

print("=== Comprehensive Evaluation Metrics ===\n")

# Calculate all metrics
metrics = {
    'Unigram': {},
    'Bigram': {},
    'Trigram': {}
}

for model_name, model, model_type in [
    ('Unigram', unigram, 'unigram'),
    ('Bigram', bigram, 'bigram'),
    ('Trigram', trigram, 'trigram')
]:
    metrics[model_name]['Perplexity'] = calculate_perplexity(model, test_corpus, model_type)
    metrics[model_name]['Cross-Entropy'] = calculate_cross_entropy(model, test_corpus, model_type)
    metrics[model_name]['BPC'] = calculate_bits_per_char(model, test_corpus, model_type)

# Display results
print(f"{'Model':<12} {'Perplexity':<15} {'Cross-Entropy':<15} {'Bits/Char':<15}")
print("-" * 60)
for model_name in ['Unigram', 'Bigram', 'Trigram']:
    print(f"{model_name:<12} "
          f"{metrics[model_name]['Perplexity']:<15.2f} "
          f"{metrics[model_name]['Cross-Entropy']:<15.4f} "
          f"{metrics[model_name]['BPC']:<15.4f}")

print("\n" + "="*60 + "\n")

# Visualize all metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

model_names = ['Unigram', 'Bigram', 'Trigram']
colors = ['blue', 'green', 'orange']

# Perplexity
ppls = [metrics[m]['Perplexity'] for m in model_names]
axes[0].bar(model_names, ppls, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_title('Perplexity\n(lower is better)', fontsize=12, weight='bold')
axes[0].set_ylabel('Value')
axes[0].grid(True, alpha=0.3, axis='y')

# Cross-Entropy
ces = [metrics[m]['Cross-Entropy'] for m in model_names]
axes[1].bar(model_names, ces, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_title('Cross-Entropy\n(lower is better)', fontsize=12, weight='bold')
axes[1].set_ylabel('Value')
axes[1].grid(True, alpha=0.3, axis='y')

# Bits per Character
bpcs = [metrics[m]['BPC'] for m in model_names]
axes[2].bar(model_names, bpcs, color=colors, alpha=0.7, edgecolor='black')
axes[2].set_title('Bits per Character\n(lower is better)', fontsize=12, weight='bold')
axes[2].set_ylabel('Value')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("指标说明：")
print("- Perplexity: 模型在每个位置平均考虑多少个词")
print("- Cross-Entropy: 平均负对数似然（信息论角度）")
print("- Bits/Char: 每个字符需要多少比特编码（压缩角度）")
print("\n关系：PPL = exp(Cross-Entropy)")
print("\n✓ 评估指标实现完成！")

## 5. 神经语言模型

### 5.1 从统计到神经

**N-gram模型的局限**：

1. **稀疏性问题**：大量n-gram从未见过
2. **泛化能力差**：无法处理相似词（"cat" vs "dog"）
3. **上下文受限**：N通常≤5，无法捕捉长距离依赖
4. **存储开销大**：需要存储所有n-gram计数

**神经语言模型的优势**：

1. **分布式表示**：词嵌入捕捉语义相似性
2. **参数共享**：通过权重矩阵，不需要存储每个n-gram
3. **泛化能力强**：相似词有相似表示
4. **灵活的上下文**：可以处理任意长度的历史

### 5.2 神经语言模型架构

**基本结构**：

```
输入词 → 词嵌入 → 循环层 → 输出层 → 概率分布
```

**数学表示**：

1. **词嵌入**：
   $$e_t = E[w_t]$$
   其中 $E \in \mathbb{R}^{|V| \times d}$ 是嵌入矩阵

2. **循环层（RNN）**：
   $$h_t = \tanh(W_h h_{t-1} + W_x e_t + b_h)$$

3. **输出层**：
   $$\hat{y}_t = \text{softmax}(W_o h_t + b_o)$$
   $$P(w_t | w_{<t}) = \hat{y}_t[w_t]$$

**训练目标**：最大化对数似然

$$\mathcal{L} = \sum_{t=1}^T \log P(w_t | w_{<t})$$

等价于最小化交叉熵损失。

In [ ]:
# 🔬 Micro Practice: Implement RNN Language Model
# Goal: Build neural language model with PyTorch
# Expected outcome: Understand neural LM architecture

class RNNLanguageModel(nn.Module):
    """
    RNN-based Language Model
    """
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        """
        Args:
            vocab_size: Size of vocabulary
            embedding_dim: Dimension of word embeddings
            hidden_dim: Dimension of hidden state
        """
        super(RNNLanguageModel, self).__init__()
        
        self.hidden_dim = hidden_dim
        
        # Word embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # RNN layer
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        
        # Output layer
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        """
        Forward pass
        
        Args:
            x: Input token indices (batch_size, seq_length)
            hidden: Initial hidden state
        
        Returns:
            output: Logits for next word prediction (batch_size, seq_length, vocab_size)
            hidden: Final hidden state
        """
        # Embed input tokens
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim)
        
        # Pass through RNN
        rnn_out, hidden = self.rnn(embedded, hidden)  # (batch_size, seq_length, hidden_dim)
        
        # Project to vocabulary
        output = self.fc(rnn_out)  # (batch_size, seq_length, vocab_size)
        
        return output, hidden
    
    def init_hidden(self, batch_size):
        """Initialize hidden state"""
        return torch.zeros(1, batch_size, self.hidden_dim)

# Build vocabulary from corpus
def build_vocab(corpus):
    """Build vocabulary and word-to-index mappings"""
    vocab = set()
    for sentence in corpus:
        vocab.update(sentence)
    
    vocab = ['<PAD>', '<START>', '<END>'] + sorted(list(vocab))
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    
    return vocab, word2idx, idx2word

# Convert sentences to indices
def sentences_to_indices(sentences, word2idx):
    """Convert sentences to token indices"""
    indices = []
    for sentence in sentences:
        sent_indices = [word2idx['<START>']] + \
                       [word2idx.get(word, word2idx['<PAD>']) for word in sentence] + \
                       [word2idx['<END>']]
        indices.append(sent_indices)
    return indices

# Prepare data
vocab, word2idx, idx2word = build_vocab(corpus)
train_indices = sentences_to_indices(corpus, word2idx)

print("=== RNN Language Model ===\n")
print(f"Vocabulary size: {len(vocab)}")
print(f"Sample vocab: {vocab[:10]}")
print(f"\nSample sentence indices:")
print(f"  Original: {corpus[0]}")
print(f"  Indices: {train_indices[0]}")
print()

# Create model
vocab_size = len(vocab)
embedding_dim = 16
hidden_dim = 32

model = RNNLanguageModel(vocab_size, embedding_dim, hidden_dim)

print(f"Model architecture:")
print(f"  Embedding: {vocab_size} → {embedding_dim}")
print(f"  RNN: {embedding_dim} → {hidden_dim}")
print(f"  Output: {hidden_dim} → {vocab_size}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")
print("\n✓ RNN语言模型创建完成！")

In [ ]:
# 🔬 Micro Practice: Train RNN Language Model
# Goal: Train neural LM and generate text
# Expected outcome: See neural LM in action

def train_rnn_lm(model, train_data, word2idx, epochs=50, lr=0.01):
    """
    Train RNN language model
    
    Args:
        model: RNN language model
        train_data: List of token index sequences
        word2idx: Word to index mapping
        epochs: Number of training epochs
        lr: Learning rate
    
    Returns:
        losses: Training loss history
    """
    criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    
    for epoch in range(epochs):
        total_loss = 0
        
        for sent_indices in train_data:
            # Prepare input and target
            # Input: <START> w1 w2 w3 ... wn
            # Target: w1 w2 w3 ... wn <END>
            input_seq = torch.tensor([sent_indices[:-1]])  # (1, seq_length)
            target_seq = torch.tensor([sent_indices[1:]])  # (1, seq_length)
            
            # Forward pass
            optimizer.zero_grad()
            output, _ = model(input_seq)  # (1, seq_length, vocab_size)
            
            # Compute loss
            loss = criterion(output.view(-1, len(word2idx)), target_seq.view(-1))
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_data)
        losses.append(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    return losses

def generate_text_rnn(model, word2idx, idx2word, start_token='<START>', max_length=20, temperature=1.0):
    """
    Generate text using RNN language model
    
    Args:
        model: Trained RNN language model
        word2idx: Word to index mapping
        idx2word: Index to word mapping
        start_token: Starting token
        max_length: Maximum generation length
        temperature: Sampling temperature (higher = more random)
    
    Returns:
        Generated sentence
    """
    model.eval()
    
    with torch.no_grad():
        # Start with start token
        current_idx = word2idx[start_token]
        generated = []
        hidden = None
        
        for _ in range(max_length):
            # Prepare input
            input_tensor = torch.tensor([[current_idx]])
            
            # Forward pass
            output, hidden = model(input_tensor, hidden)
            
            # Get probabilities for next word
            logits = output[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=0)
            
            # Sample next word
            next_idx = torch.multinomial(probs, 1).item()
            
            # Stop if end token
            if idx2word[next_idx] == '<END>':
                break
            
            # Skip special tokens in output
            if idx2word[next_idx] not in ['<PAD>', '<START>']:
                generated.append(idx2word[next_idx])
            
            current_idx = next_idx
    
    return ' '.join(generated)

# Train model
print("Training RNN Language Model...\n")
losses = train_rnn_lm(model, train_indices, word2idx, epochs=100, lr=0.01)

print("\n" + "="*50 + "\n")

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(losses, linewidth=2)
plt.title('RNN Language Model Training Loss', fontsize=14, weight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Generate text samples
print("\nGenerated Text Samples:\n")
for i in range(5):
    generated = generate_text_rnn(model, word2idx, idx2word, temperature=0.8)
    print(f"  {i+1}. {generated}")

print("\n观察：神经语言模型生成的文本更加流畅")
print("✓ RNN语言模型训练完成！")

In [ ]:
# 🔬 Micro Practice: Compare Neural LM with N-gram Models
# Goal: Evaluate neural LM vs statistical models
# Expected outcome: Understand advantages of neural approach

def calculate_perplexity_neural(model, test_data, word2idx):
    """
    Calculate perplexity for neural language model
    
    Args:
        model: Trained neural language model
        test_data: List of token index sequences
        word2idx: Word to index mapping
    
    Returns:
        Perplexity value
    """
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'], reduction='sum')
    
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for sent_indices in test_data:
            input_seq = torch.tensor([sent_indices[:-1]])
            target_seq = torch.tensor([sent_indices[1:]])
            
            output, _ = model(input_seq)
            loss = criterion(output.view(-1, len(word2idx)), target_seq.view(-1))
            
            total_loss += loss.item()
            total_tokens += len(sent_indices) - 1
    
    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)
    
    return perplexity

# Prepare test data
test_indices = sentences_to_indices(test_corpus, word2idx)

# Calculate perplexity for neural model
ppl_rnn = calculate_perplexity_neural(model, test_indices, word2idx)

print("=== Model Comparison: Statistical vs Neural ===\n")
print(f"{'Model':<15} {'Perplexity':<15} {'Parameters':<15}")
print("-" * 45)
print(f"{'Unigram':<15} {ppl_unigram:<15.2f} {'~' + str(len(unigram.vocab)):<15}")
print(f"{'Bigram':<15} {ppl_bigram:<15.2f} {'~' + str(sum(len(v) for v in bigram.bigram_counts.values())):<15}")
print(f"{'Trigram':<15} {ppl_trigram:<15.2f} {'~' + str(sum(sum(len(v2) for v2 in v1.values()) for v1 in trigram.trigram_counts.values())):<15}")
print(f"{'RNN':<15} {ppl_rnn:<15.2f} {str(sum(p.numel() for p in model.parameters())):<15}")

print("\n" + "="*45 + "\n")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Perplexity comparison
model_names = ['Unigram', 'Bigram', 'Trigram', 'RNN']
perplexities = [ppl_unigram, ppl_bigram, ppl_trigram, ppl_rnn]
colors = ['blue', 'green', 'orange', 'red']

bars = axes[0].bar(model_names, perplexities, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_title('Perplexity Comparison', fontsize=12, weight='bold')
axes[0].set_ylabel('Perplexity (lower is better)')
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, ppl in zip(bars, perplexities):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{ppl:.1f}',
                ha='center', va='bottom', fontsize=10, weight='bold')

# Generation quality comparison (sample diversity)
print("Generation Samples Comparison:\n")

print("Bigram:")
for i in range(3):
    print(f"  {i+1}. {bigram.generate(10)}")

print("\nRNN (temp=0.8):")
for i in range(3):
    print(f"  {i+1}. {generate_text_rnn(model, word2idx, idx2word, temperature=0.8)}")

print("\nRNN (temp=1.2 - more diverse):")
for i in range(3):
    print(f"  {i+1}. {generate_text_rnn(model, word2idx, idx2word, temperature=1.2)}")

# Visualize generation diversity
axes[1].text(0.5, 0.9, 'Neural LM Advantages:', ha='center', va='top', 
             fontsize=14, weight='bold', transform=axes[1].transAxes)

advantages = [
    '✓ 分布式表示（词嵌入）',
    '✓ 参数共享（泛化能力强）',
    '✓ 无稀疏性问题',
    '✓ 可处理任意长度上下文',
    '✓ 可控的生成（temperature）'
]

for i, adv in enumerate(advantages):
    axes[1].text(0.1, 0.75 - i*0.12, adv, ha='left', va='top',
                fontsize=11, transform=axes[1].transAxes)

axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\n核心观察：")
print("- 神经LM在小数据上可能不如n-gram（需要更多数据）")
print("- 但神经LM泛化能力更强，可扩展性更好")
print("- 实践中：大规模数据 + 神经LM = 最佳性能")
print("\n✓ 模型比较完成！")

## 6. 现代神经架构

### 6.1 LSTM 语言模型

**RNN的局限**：梯度消失问题限制了长距离依赖的学习。

**LSTM的优势**：
- 门控机制控制信息流
- 细胞状态提供梯度高速公路
- 更好地捕捉长期依赖

**LSTM语言模型架构**：

```
输入词 → 词嵌入 → LSTM层 → 输出层 → 概率分布
```

与RNN的主要区别：用LSTM单元替换简单的RNN单元。

In [ ]:
# 🔬 Micro Practice: Implement LSTM Language Model
# Goal: Build LSTM-based language model
# Expected outcome: Compare LSTM with RNN

class LSTMLanguageModel(nn.Module):
    """
    LSTM-based Language Model
    """
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        """
        Args:
            vocab_size: Size of vocabulary
            embedding_dim: Dimension of word embeddings
            hidden_dim: Dimension of hidden state
            num_layers: Number of LSTM layers
        """
        super(LSTMLanguageModel, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Word embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # LSTM layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        
        # Output layer
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        """
        Forward pass
        
        Args:
            x: Input token indices (batch_size, seq_length)
            hidden: Initial hidden state (h, c)
        
        Returns:
            output: Logits for next word prediction
            hidden: Final hidden state (h, c)
        """
        # Embed input tokens
        embedded = self.embedding(x)
        
        # Pass through LSTM
        lstm_out, hidden = self.lstm(embedded, hidden)
        
        # Project to vocabulary
        output = self.fc(lstm_out)
        
        return output, hidden
    
    def init_hidden(self, batch_size):
        """Initialize hidden state"""
        h = torch.zeros(self.num_layers, batch_size, self.hidden_dim)
        c = torch.zeros(self.num_layers, batch_size, self.hidden_dim)
        return (h, c)

# Create LSTM model
lstm_model = LSTMLanguageModel(vocab_size, embedding_dim=16, hidden_dim=32, num_layers=2)

print("=== LSTM Language Model ===\n")
print(f"Model architecture:")
print(f"  Embedding: {vocab_size} → {16}")
print(f"  LSTM: {16} → {32} (2 layers)")
print(f"  Output: {32} → {vocab_size}")
print(f"\nTotal parameters: {sum(p.numel() for p in lstm_model.parameters())}")

# Train LSTM model
print("\nTraining LSTM Language Model...\n")
lstm_losses = train_rnn_lm(lstm_model, train_indices, word2idx, epochs=100, lr=0.01)

print("\n✓ LSTM语言模型训练完成！")

### 6.2 Transformer 语言模型

**Transformer的优势**：
- 自注意力机制：并行处理所有位置
- 无梯度消失问题
- 更好地捕捉长距离依赖
- 可扩展性强（GPT系列的基础）

**Decoder-only架构**（GPT风格）：

```
输入词 → 词嵌入 + 位置编码 → Transformer Decoder层 → 输出层
```

**关键特性**：
- **因果掩码（Causal Masking）**：确保位置i只能看到i之前的词
- **多头自注意力**：捕捉不同类型的依赖关系
- **前馈网络**：增加模型容量

In [ ]:
# 🔬 Micro Practice: Implement Transformer Language Model
# Goal: Build simplified Transformer LM
# Expected outcome: Understand modern LM architecture

class TransformerLanguageModel(nn.Module):
    """
    Simplified Transformer Language Model (GPT-style)
    """
    
    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, max_seq_length=50):
        """
        Args:
            vocab_size: Size of vocabulary
            embedding_dim: Dimension of embeddings
            num_heads: Number of attention heads
            num_layers: Number of transformer layers
            max_seq_length: Maximum sequence length
        """
        super(TransformerLanguageModel, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.max_seq_length = max_seq_length
        
        # Token embedding
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Positional embedding
        self.position_embedding = nn.Embedding(max_seq_length, embedding_dim)
        
        # Transformer decoder layers
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=embedding_dim * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers)
        
        # Output layer
        self.fc = nn.Linear(embedding_dim, vocab_size)
    
    def generate_square_subsequent_mask(self, sz):
        """Generate causal mask"""
        mask = torch.triu(torch.ones(sz, sz), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: Input token indices (batch_size, seq_length)
        
        Returns:
            output: Logits for next word prediction
        """
        batch_size, seq_length = x.shape
        
        # Token embeddings
        token_emb = self.token_embedding(x)  # (batch_size, seq_length, embedding_dim)
        
        # Positional embeddings
        positions = torch.arange(seq_length, device=x.device).unsqueeze(0)
        pos_emb = self.position_embedding(positions)
        
        # Combine embeddings
        embeddings = token_emb + pos_emb
        
        # Create causal mask
        mask = self.generate_square_subsequent_mask(seq_length).to(x.device)
        
        # Pass through transformer (using self as memory for decoder-only)
        transformer_out = self.transformer(embeddings, embeddings, tgt_mask=mask)
        
        # Project to vocabulary
        output = self.fc(transformer_out)
        
        return output

# Create Transformer model
transformer_model = TransformerLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=32,
    num_heads=2,
    num_layers=2
)

print("=== Transformer Language Model ===\n")
print(f"Model architecture:")
print(f"  Token Embedding: {vocab_size} → {32}")
print(f"  Position Embedding: {50} → {32}")
print(f"  Transformer: 2 layers, 2 heads")
print(f"  Output: {32} → {vocab_size}")
print(f"\nTotal parameters: {sum(p.numel() for p in transformer_model.parameters())}")

# Train Transformer model (simplified training function)
def train_transformer_lm(model, train_data, word2idx, epochs=50, lr=0.001):
    """Train Transformer language model"""
    criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    
    for epoch in range(epochs):
        total_loss = 0
        
        for sent_indices in train_data:
            input_seq = torch.tensor([sent_indices[:-1]])
            target_seq = torch.tensor([sent_indices[1:]])
            
            optimizer.zero_grad()
            output = model(input_seq)
            
            loss = criterion(output.view(-1, len(word2idx)), target_seq.view(-1))
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_data)
        losses.append(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    return losses

print("\nTraining Transformer Language Model...\n")
transformer_losses = train_transformer_lm(transformer_model, train_indices, word2idx, epochs=100, lr=0.001)

print("\n✓ Transformer语言模型训练完成！")

In [ ]:
# 🔬 Micro Practice: Compare Neural Architectures
# Goal: Benchmark RNN vs LSTM vs Transformer
# Expected outcome: Understand trade-offs

# Calculate perplexity for all neural models
ppl_lstm = calculate_perplexity_neural(lstm_model, test_indices, word2idx)

# For Transformer, we need a modified perplexity function
def calculate_perplexity_transformer(model, test_data, word2idx):
    """Calculate perplexity for Transformer model"""
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'], reduction='sum')
    
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for sent_indices in test_data:
            input_seq = torch.tensor([sent_indices[:-1]])
            target_seq = torch.tensor([sent_indices[1:]])
            
            output = model(input_seq)
            loss = criterion(output.view(-1, len(word2idx)), target_seq.view(-1))
            
            total_loss += loss.item()
            total_tokens += len(sent_indices) - 1
    
    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)
    
    return perplexity

ppl_transformer = calculate_perplexity_transformer(transformer_model, test_indices, word2idx)

print("=== Neural Architecture Comparison ===\n")
print(f"{'Architecture':<15} {'Perplexity':<15} {'Parameters':<15} {'Training Time':<15}")
print("-" * 60)
print(f"{'RNN':<15} {ppl_rnn:<15.2f} {sum(p.numel() for p in model.parameters()):<15} {'Fast':<15}")
print(f"{'LSTM':<15} {ppl_lstm:<15.2f} {sum(p.numel() for p in lstm_model.parameters()):<15} {'Medium':<15}")
print(f"{'Transformer':<15} {ppl_transformer:<15.2f} {sum(p.numel() for p in transformer_model.parameters()):<15} {'Slow':<15}")

print("\n" + "="*60 + "\n")

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Training loss curves
axes[0, 0].plot(losses, label='RNN', linewidth=2, alpha=0.7)
axes[0, 0].plot(lstm_losses, label='LSTM', linewidth=2, alpha=0.7)
axes[0, 0].plot(transformer_losses, label='Transformer', linewidth=2, alpha=0.7)
axes[0, 0].set_title('Training Loss Curves', fontsize=12, weight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Perplexity comparison
architectures = ['RNN', 'LSTM', 'Transformer']
perplexities = [ppl_rnn, ppl_lstm, ppl_transformer]
colors = ['blue', 'green', 'red']

bars = axes[0, 1].bar(architectures, perplexities, color=colors, alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Perplexity Comparison', fontsize=12, weight='bold')
axes[0, 1].set_ylabel('Perplexity (lower is better)')
axes[0, 1].grid(True, alpha=0.3, axis='y')

for bar, ppl in zip(bars, perplexities):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{ppl:.1f}',
                    ha='center', va='bottom', fontsize=10, weight='bold')

# 3. Parameter count
param_counts = [
    sum(p.numel() for p in model.parameters()),
    sum(p.numel() for p in lstm_model.parameters()),
    sum(p.numel() for p in transformer_model.parameters())
]

axes[1, 0].bar(architectures, param_counts, color=colors, alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Model Size (Parameters)', fontsize=12, weight='bold')
axes[1, 0].set_ylabel('Number of Parameters')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Architecture characteristics
axes[1, 1].axis('off')
axes[1, 1].text(0.5, 0.95, 'Architecture Characteristics', 
                ha='center', va='top', fontsize=14, weight='bold',
                transform=axes[1, 1].transAxes)

characteristics = [
    'RNN:',
    '  ✓ 简单、参数少',
    '  ✗ 梯度消失、难训练',
    '',
    'LSTM:',
    '  ✓ 解决梯度消失',
    '  ✓ 捕捉长期依赖',
    '  ✗ 串行处理、较慢',
    '',
    'Transformer:',
    '  ✓ 并行处理',
    '  ✓ 最佳性能',
    '  ✗ 参数多、需要更多数据'
]

y_pos = 0.85
for char in characteristics:
    axes[1, 1].text(0.1, y_pos, char, ha='left', va='top',
                   fontsize=10, transform=axes[1, 1].transAxes)
    y_pos -= 0.06

plt.tight_layout()
plt.show()

print("核心观察：")
print("- RNN: 简单但性能受限")
print("- LSTM: 平衡性能和复杂度")
print("- Transformer: 最佳性能，但需要更多资源")
print("- 实践中: Transformer主导大规模语言模型（GPT、BERT等）")
print("\n✓ 架构比较完成！")

## 7. 文本生成策略

### 7.1 解码方法

给定语言模型 $P(w_t | w_{<t})$，如何生成文本？

**常见策略**：

1. **贪心解码（Greedy Decoding）**
   - 每步选择概率最高的词
   - $$w_t = \arg\max_w P(w | w_{<t})$$
   - 优点：快速、确定性
   - 缺点：可能陷入局部最优，生成重复

2. **束搜索（Beam Search）**
   - 保留top-k个候选序列
   - 平衡质量和多样性
   - 优点：比贪心更好的全局质量
   - 缺点：仍可能重复，计算量大

3. **随机采样（Sampling）**
   - 按概率分布随机采样
   - $$w_t \sim P(w | w_{<t})$$
   - 优点：多样性高
   - 缺点：可能生成低质量文本

4. **Top-k 采样**
   - 只从概率最高的k个词中采样
   - 过滤低概率词
   - 优点：平衡质量和多样性

5. **Nucleus (Top-p) 采样**
   - 从累积概率达到p的最小词集中采样
   - 动态调整候选集大小
   - 优点：更灵活，适应不同情况

6. **温度采样（Temperature Scaling）**
   - 调整概率分布的"尖锐度"
   - $$P'(w) = \frac{\exp(\text{logit}_w / T)}{\sum_w \exp(\text{logit}_w / T)}$$
   - T < 1：更确定（尖锐）
   - T > 1：更随机（平滑）
   - T = 1：原始分布

In [ ]:
# 🔬 Micro Practice: Implement Generation Strategies
# Goal: Build different decoding methods
# Expected outcome: Understand generation trade-offs

def generate_greedy(model, word2idx, idx2word, start_token='<START>', max_length=20):
    """Greedy decoding: always pick most likely word"""
    model.eval()
    
    with torch.no_grad():
        current_idx = word2idx[start_token]
        generated = []
        hidden = None
        
        for _ in range(max_length):
            input_tensor = torch.tensor([[current_idx]])
            output, hidden = model(input_tensor, hidden)
            
            # Pick most likely word
            logits = output[0, -1, :]
            next_idx = torch.argmax(logits).item()
            
            if idx2word[next_idx] == '<END>':
                break
            
            if idx2word[next_idx] not in ['<PAD>', '<START>']:
                generated.append(idx2word[next_idx])
            
            current_idx = next_idx
    
    return ' '.join(generated)

def generate_topk(model, word2idx, idx2word, k=5, start_token='<START>', max_length=20, temperature=1.0):
    """Top-k sampling: sample from top k most likely words"""
    model.eval()
    
    with torch.no_grad():
        current_idx = word2idx[start_token]
        generated = []
        hidden = None
        
        for _ in range(max_length):
            input_tensor = torch.tensor([[current_idx]])
            output, hidden = model(input_tensor, hidden)
            
            # Apply temperature
            logits = output[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=0)
            
            # Get top-k
            topk_probs, topk_indices = torch.topk(probs, k)
            topk_probs = topk_probs / topk_probs.sum()  # Renormalize
            
            # Sample from top-k
            next_idx = topk_indices[torch.multinomial(topk_probs, 1)].item()
            
            if idx2word[next_idx] == '<END>':
                break
            
            if idx2word[next_idx] not in ['<PAD>', '<START>']:
                generated.append(idx2word[next_idx])
            
            current_idx = next_idx
    
    return ' '.join(generated)

def generate_nucleus(model, word2idx, idx2word, p=0.9, start_token='<START>', max_length=20, temperature=1.0):
    """Nucleus (top-p) sampling: sample from smallest set with cumulative prob >= p"""
    model.eval()
    
    with torch.no_grad():
        current_idx = word2idx[start_token]
        generated = []
        hidden = None
        
        for _ in range(max_length):
            input_tensor = torch.tensor([[current_idx]])
            output, hidden = model(input_tensor, hidden)
            
            # Apply temperature
            logits = output[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=0)
            
            # Sort probabilities
            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            
            # Find nucleus (smallest set with cumulative prob >= p)
            cumsum_probs = torch.cumsum(sorted_probs, dim=0)
            nucleus_size = (cumsum_probs <= p).sum().item() + 1
            
            # Sample from nucleus
            nucleus_probs = sorted_probs[:nucleus_size]
            nucleus_indices = sorted_indices[:nucleus_size]
            nucleus_probs = nucleus_probs / nucleus_probs.sum()  # Renormalize
            
            next_idx = nucleus_indices[torch.multinomial(nucleus_probs, 1)].item()
            
            if idx2word[next_idx] == '<END>':
                break
            
            if idx2word[next_idx] not in ['<PAD>', '<START>']:
                generated.append(idx2word[next_idx])
            
            current_idx = next_idx
    
    return ' '.join(generated)

print("=== Text Generation Strategies ===\n")

# Use LSTM model for generation
print("1. Greedy Decoding (deterministic):")
for i in range(3):
    text = generate_greedy(lstm_model, word2idx, idx2word)
    print(f"   {i+1}. {text}")

print("\n2. Top-k Sampling (k=3, temp=0.8):")
for i in range(3):
    text = generate_topk(lstm_model, word2idx, idx2word, k=3, temperature=0.8)
    print(f"   {i+1}. {text}")

print("\n3. Top-k Sampling (k=5, temp=1.2 - more diverse):")
for i in range(3):
    text = generate_topk(lstm_model, word2idx, idx2word, k=5, temperature=1.2)
    print(f"   {i+1}. {text}")

print("\n4. Nucleus Sampling (p=0.9, temp=1.0):")
for i in range(3):
    text = generate_nucleus(lstm_model, word2idx, idx2word, p=0.9, temperature=1.0)
    print(f"   {i+1}. {text}")

print("\n5. Nucleus Sampling (p=0.95, temp=0.7 - more focused):")
for i in range(3):
    text = generate_nucleus(lstm_model, word2idx, idx2word, p=0.95, temperature=0.7)
    print(f"   {i+1}. {text}")

print("\n✓ 生成策略实现完成！")

In [ ]:
# 🔬 Micro Practice: Visualize Generation Strategy Effects
# Goal: Compare quality vs diversity trade-offs
# Expected outcome: Understand when to use which strategy

# Generate multiple samples with different strategies
def analyze_generation_diversity(model, word2idx, idx2word, n_samples=10):
    """Analyze diversity of different generation strategies"""
    
    strategies = {
        'Greedy': lambda: generate_greedy(model, word2idx, idx2word),
        'Top-k (k=3)': lambda: generate_topk(model, word2idx, idx2word, k=3, temperature=0.8),
        'Top-k (k=5)': lambda: generate_topk(model, word2idx, idx2word, k=5, temperature=1.0),
        'Nucleus (p=0.9)': lambda: generate_nucleus(model, word2idx, idx2word, p=0.9, temperature=1.0),
    }
    
    results = {}
    
    for name, gen_func in strategies.items():
        samples = [gen_func() for _ in range(n_samples)]
        
        # Calculate diversity metrics
        unique_samples = len(set(samples))
        avg_length = np.mean([len(s.split()) for s in samples])
        
        results[name] = {
            'samples': samples,
            'unique': unique_samples,
            'avg_length': avg_length,
            'diversity': unique_samples / n_samples
        }
    
    return results

print("=== Generation Strategy Analysis ===\n")
results = analyze_generation_diversity(lstm_model, word2idx, idx2word, n_samples=10)

# Display results
print(f"{'Strategy':<20} {'Unique/Total':<15} {'Diversity':<15} {'Avg Length':<15}")
print("-" * 65)
for name, data in results.items():
    print(f"{name:<20} {data['unique']}/10{'':<10} {data['diversity']:<15.2f} {data['avg_length']:<15.1f}")

print("\n" + "="*65 + "\n")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Diversity comparison
strategies = list(results.keys())
diversities = [results[s]['diversity'] for s in strategies]
colors = ['red', 'orange', 'green', 'blue']

bars = axes[0].bar(strategies, diversities, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_title('Generation Diversity', fontsize=12, weight='bold')
axes[0].set_ylabel('Diversity (unique/total)')
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=15)

for bar, div in zip(bars, diversities):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{div:.2f}',
                ha='center', va='bottom', fontsize=10, weight='bold')

# Strategy recommendations
axes[1].axis('off')
axes[1].text(0.5, 0.95, 'Strategy Selection Guide', 
            ha='center', va='top', fontsize=14, weight='bold',
            transform=axes[1].transAxes)

recommendations = [
    '贪心解码 (Greedy):',
    '  • 需要确定性输出',
    '  • 翻译、摘要等任务',
    '',
    'Top-k 采样:',
    '  • 平衡质量和多样性',
    '  • 对话系统、创意写作',
    '',
    'Nucleus 采样:',
    '  • 最灵活的方法',
    '  • 现代LLM的标准选择',
    '',
    '温度调节:',
    '  • T < 1: 更保守、更准确',
    '  • T > 1: 更创意、更多样'
]

y_pos = 0.85
for rec in recommendations:
    axes[1].text(0.05, y_pos, rec, ha='left', va='top',
                fontsize=10, transform=axes[1].transAxes)
    y_pos -= 0.055

plt.tight_layout()
plt.show()

print("核心观察：")
print("- 贪心解码：确定性强，但可能重复")
print("- Top-k：平衡质量和多样性")
print("- Nucleus：最灵活，适应不同情况")
print("- 实践中：Nucleus + 温度调节是最常用组合")
print("\n✓ 生成策略分析��成！")

## 8. 实践应用与总结

### 8.1 完整的语言建模流程

从数据到生成的完整pipeline：

1. **数据准备**：分词、构建词表
2. **模型选择**：根据数据量和任务选择架构
3. **训练**：优化交叉熵损失
4. **评估**：计算困惑度
5. **生成**：选择合适的解码策略

In [ ]:
# 🎯 Comprehensive Example: Complete Language Modeling Pipeline
# Goal: Build end-to-end language modeling system
# Expected outcome: Understand full workflow

class LanguageModelingPipeline:
    """
    Complete language modeling pipeline
    """
    
    def __init__(self, model_type='lstm', embedding_dim=32, hidden_dim=64):
        """
        Args:
            model_type: 'rnn', 'lstm', or 'transformer'
            embedding_dim: Embedding dimension
            hidden_dim: Hidden dimension
        """
        self.model_type = model_type
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.model = None
        self.vocab = None
        self.word2idx = None
        self.idx2word = None
    
    def prepare_data(self, corpus):
        """Step 1: Prepare data"""
        print("Step 1: Preparing data...")
        
        # Build vocabulary
        self.vocab, self.word2idx, self.idx2word = build_vocab(corpus)
        
        # Convert to indices
        train_indices = sentences_to_indices(corpus, self.word2idx)
        
        print(f"  ✓ Vocabulary size: {len(self.vocab)}")
        print(f"  ✓ Training samples: {len(train_indices)}")
        
        return train_indices
    
    def build_model(self):
        """Step 2: Build model"""
        print("\nStep 2: Building model...")
        
        vocab_size = len(self.vocab)
        
        if self.model_type == 'rnn':
            self.model = RNNLanguageModel(vocab_size, self.embedding_dim, self.hidden_dim)
        elif self.model_type == 'lstm':
            self.model = LSTMLanguageModel(vocab_size, self.embedding_dim, self.hidden_dim, num_layers=2)
        elif self.model_type == 'transformer':
            self.model = TransformerLanguageModel(vocab_size, self.embedding_dim, num_heads=4, num_layers=2)
        
        print(f"  ✓ Model type: {self.model_type.upper()}")
        print(f"  ✓ Parameters: {sum(p.numel() for p in self.model.parameters())}")
    
    def train(self, train_data, epochs=100, lr=0.01):
        """Step 3: Train model"""
        print("\nStep 3: Training model...")
        
        if self.model_type == 'transformer':
            losses = train_transformer_lm(self.model, train_data, self.word2idx, epochs, lr)
        else:
            losses = train_rnn_lm(self.model, train_data, self.word2idx, epochs, lr)
        
        print(f"  ✓ Training complete")
        print(f"  ✓ Final loss: {losses[-1]:.4f}")
        
        return losses
    
    def evaluate(self, test_data):
        """Step 4: Evaluate model"""
        print("\nStep 4: Evaluating model...")
        
        if self.model_type == 'transformer':
            ppl = calculate_perplexity_transformer(self.model, test_data, self.word2idx)
        else:
            ppl = calculate_perplexity_neural(self.model, test_data, self.word2idx)
        
        print(f"  ✓ Perplexity: {ppl:.2f}")
        
        return ppl
    
    def generate(self, strategy='nucleus', n_samples=3, **kwargs):
        """Step 5: Generate text"""
        print(f"\nStep 5: Generating text ({strategy})...")
        
        samples = []
        
        for i in range(n_samples):
            if strategy == 'greedy':
                text = generate_greedy(self.model, self.word2idx, self.idx2word)
            elif strategy == 'topk':
                text = generate_topk(self.model, self.word2idx, self.idx2word, **kwargs)
            elif strategy == 'nucleus':
                text = generate_nucleus(self.model, self.word2idx, self.idx2word, **kwargs)
            else:
                text = generate_text_rnn(self.model, self.word2idx, self.idx2word, **kwargs)
            
            samples.append(text)
            print(f"  {i+1}. {text}")
        
        return samples

# Run complete pipeline
print("="*70)
print("COMPLETE LANGUAGE MODELING PIPELINE")
print("="*70 + "\n")

# Initialize pipeline
pipeline = LanguageModelingPipeline(model_type='lstm', embedding_dim=32, hidden_dim=64)

# Run all steps
train_data = pipeline.prepare_data(corpus)
pipeline.build_model()
losses = pipeline.train(train_data, epochs=100, lr=0.01)
test_data = sentences_to_indices(test_corpus, pipeline.word2idx)
perplexity = pipeline.evaluate(test_data)
samples = pipeline.generate(strategy='nucleus', n_samples=5, p=0.9, temperature=0.8)

print("\n" + "="*70)
print("✓ PIPELINE COMPLETE!")
print("="*70)

### 8.2 常见问题与调试

#### Q1: 为什么使用困惑度而不是准确率？

**A:** 语言建模是预测概率分布，不是分类任务。困惑度衡量模型对测试数据的"惊讶程度"，更适合评估生成质量。

#### Q2: 如何处理OOV（Out-of-Vocabulary）词？

**A:** 常见方法：
- 使用 `<UNK>` 标记替换低频词
- 子词分词（BPE、WordPiece）
- 字符级模型

#### Q3: 语言建模和分类有什么区别？

**A:**
- **语言建模**：预测下一个词的概率分布，无监督学习
- **分类**：预测固定类别，有监督学习
- 语言建模可以用于预训练，然后微调到分类任务

#### Q4: 需要多少数据？

**A:**
- **N-gram**：几千到几万句子
- **RNN/LSTM**：几万到几十万句子
- **Transformer**：百万级以上（GPT-3用了45TB文本）
- 数据越多，模型越大，性能越好

#### Q5: 语言建模与预训练的关系？

**A:**
- **语言建模是预训练的核心任务**
- GPT系列：单向语言建模（预测下一个词）
- BERT：掩码语言建模（预测被遮盖的词）
- 预训练学到的表示可以迁移到下游任务

#### Q6: 如何选择生成策略？

**A:**
- **确定性任务**（翻译、摘要）：贪心或束搜索
- **创意任务**（对话、写作）：Top-k 或 Nucleus
- **调试**：从贪心开始，逐步增加随机性
- **生产环境**：Nucleus (p=0.9) + Temperature (0.7-1.0)

#### Q7: 为什么我的模型生成重复文本？

**A:** 可能原因：
- 使用贪心解码 → 改用采样
- 温度太低 → 增加温度
- 训练数据太少 → 增加数据或正则化
- 模型过拟合 → 添加dropout

### 8.3 本章总结

#### 核心要点

1. **语言建模基础**
   - 目标：建模 $P(w_1, w_2, ..., w_n)$
   - 应用：文本生成、预训练、评估

2. **统计语言模型**
   - Unigram：词频模型，忽略上下文
   - N-gram：马尔可夫假设，有限上下文
   - 问题：稀疏性、存储开销、泛化能力差

3. **评估指标**
   - 困惑度（Perplexity）：越低越好
   - 交叉熵：信息论角度
   - 关系：PPL = exp(Cross-Entropy)

4. **神经语言模型**
   - 词嵌入：分布式表示
   - RNN/LSTM：处理序列，捕捉依赖
   - Transformer：并行处理，最佳性能

5. **文本生成策略**
   - 贪心：确定性，可能重复
   - Top-k/Nucleus：平衡质量和多样性
   - 温度：控制随机性

#### 演进路线

```
统计语言模型 (1990s)
    ↓
神经语言模型 (2000s)
    ↓
RNN/LSTM语言模型 (2010s)
    ↓
Transformer语言模型 (2017+)
    ↓
大规模预训练模型 (GPT, BERT)
    ↓
超大规模语言模型 (GPT-3, GPT-4)
```

#### 实践建议

1. **数据准备**
   - 清洗文本，统一格式
   - 合理设置词表大小
   - 划分训练/验证/测试集

2. **模型选择**
   - 小数据：N-gram + 平滑
   - 中等数据：LSTM
   - 大数据：Transformer

3. **训练技巧**
   - 使用学习率调度
   - 梯度裁剪防止爆炸
   - 早停防止过拟合

4. **生成调优**
   - 从贪心开始调试
   - 调整温度和采样参数
   - 评估多样性和质量

### 8.4 下一章预告

**Module 4.2: BERT架构与预训练**
- 双向语言模型
- 掩码语言建模（MLM）
- Next Sentence Prediction（NSP）
- BERT的应用

### 💡 思考题

1. 为什么Transformer在大规模语言建模中优于LSTM？

2. 如果训练数据包含多种语言，语言模型会学到什么？

3. 如何设计一个能够生成代码的语言模型？

4. 困惑度为1意味着什么？实际中能达到吗？

5. 为什么现代LLM（如GPT-4）使用Nucleus采样而不是贪心解码？

## 思考题参考答案

### 问题 1：什么是语言建模？为什么它对现代 NLP 如此重要？

**语言建模的定义**

语言模型（Language Model, LM）是对自然语言序列的概率分布进行建模的系统。其核心任务是：给定历史词序列，预测下一个词出现的概率。

数学表示：对于句子 $w_1, w_2, \ldots, w_n$，语言模型计算：

$$P(w_1, w_2, \ldots, w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, w_2, \ldots, w_{i-1})$$

**为什么重要？**

| 重要性维度 | 具体说明 |
|-----------|---------|
| 基础任务地位 | 语言建模是 NLP 所有任务的底层基础，几乎所有 NLP 系统都依赖对语言概率分布的理解 |
| 无监督学习 | 只需要原始文本即可训练，不需要人工标注，可以利用互联网上海量数据 |
| 迁移学习基础 | BERT、GPT 等大模型均以语言建模为预训练目标，学到的表示可迁移到下游任务 |
| 应用广泛 | 文本生成、语音识别、机器翻译、拼写纠错等都依赖语言模型的概率评估 |
| 评估语言理解 | 模型的困惑度（Perplexity）直接反映其对语言规律的掌握程度 |

**核心价值**：语言建模把"理解语言"转化为"预测概率"这一可优化的数学问题，是连接人类语言与机器计算的桥梁。

---

### 问题 2：N-gram 模型的优缺点是什么？平滑技术如何解决其核心问题？

**N-gram 模型原理**

N-gram 模型利用马尔可夫假设，用前 $n-1$ 个词来预测当前词：

$$P(w_i \mid w_1, \ldots, w_{i-1}) \approx P(w_i \mid w_{i-(n-1)}, \ldots, w_{i-1})$$

**优缺点对比**

| 特性 | 优点 | 缺点 |
|------|------|------|
| 计算效率 | 训练快速，推理高效 | 存储空间随 n 指数增长 |
| 可解释性 | 直观透明，易于调试 | 无法解释为何某个 n-gram 更合理 |
| 上下文长度 | 实现简单 | 只能捕获短距离依赖（通常 n≤5） |
| 数据需求 | 小数据即可工作 | 面临数据稀疏问题（很多 n-gram 从未出现） |
| 泛化能力 | — | 无法泛化到未见过的词序列 |

**平滑技术解决稀疏性问题**

核心问题：大量 n-gram 在训练集中未出现，导致概率为 0，整个句子概率为 0。

常见平滑方法：

1. **加一平滑（Laplace 平滑）**：对所有 n-gram 计数加 1
   $$P(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i) + 1}{C(w_{i-1}) + |V|}$$

2. **Kneser-Ney 平滑**：基于词的分布多样性而非频率，是实践中最有效的方法

3. **插值平滑**：将高阶与低阶 N-gram 的概率加权组合
   $$P(w_i \mid w_{i-2}, w_{i-1}) = \lambda_3 P_3 + \lambda_2 P_2 + \lambda_1 P_1$$

4. **退避（Back-off）**：高阶 n-gram 未见时，退回到低阶 n-gram 估计

平滑技术的本质：将一部分概率质量从高频 n-gram 转移给未见 n-gram，保证所有序列都有非零概率。

---

### 问题 3：神经语言模型如何解决 N-gram 模型的稀疏性问题？

**N-gram 的根本局限**

N-gram 依赖离散的词符号匹配，"the cat sat" 与 "the dog sat" 在 N-gram 看来完全无关，无法共享统计信息。

**神经语言模型的解决方案**

1. **词嵌入（Word Embeddings）**：将离散词映射为连续向量空间
   - 语义相似的词在向量空间中距离相近
   - "cat" 和 "dog" 的向量相似，因此学到 "cat sat" 可以帮助预测 "dog sat"

2. **参数共享**：神经网络的权重对所有词序列共享
   - N-gram 每个组合单独存储，参数随词汇量指数增长
   - 神经网络参数量固定，不随词汇量指数增长

3. **长距离依赖**：
   - RNN/LSTM：通过隐藏状态传递信息，理论上可处理任意长度上下文
   - Transformer：通过自注意力机制直接建立任意位置间的依赖关系

4. **泛化能力**：模型学习到通用语言规律而非记忆特定组合

**架构演进对比**

$$\text{N-gram} \xrightarrow{\text{词嵌入}} \text{前馈神经LM} \xrightarrow{\text{序列建模}} \text{RNN-LM} \xrightarrow{\text{注意力机制}} \text{Transformer-LM}$$

神经模型的核心优势在于：用**连续向量表示**替代**离散符号**，实现了从"记忆"到"泛化"的转变。

---

### 问题 4：困惑度（Perplexity）是什么？它与交叉熵损失有什么关系？

**困惑度的定义**

困惑度衡量语言模型对测试集的预测能力，直观理解为"模型在每一步预测时平均需要在多少个候选词中做选择"：

$$\text{PPL} = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log P(w_i \mid w_1, \ldots, w_{i-1})\right)$$

**与交叉熵损失的关系**

$$\text{PPL} = e^{\mathcal{L}_{\text{CE}}}$$

其中 $\mathcal{L}_{\text{CE}} = -\frac{1}{N}\sum_{i=1}^{N} \log P(w_i \mid \text{context})$ 是平均交叉熵损失。

**直观解读**

| 困惑度值 | 含义 |
|---------|------|
| PPL = 1 | 完美预测，模型完全确定下一个词 |
| PPL = |V| | 随机猜测，相当于均匀分布 |
| PPL 越小 | 模型越好，预测越准确 |

**实际参考值**

- 英文 N-gram 模型：PPL ≈ 100-1000
- LSTM 语言模型：PPL ≈ 50-100
- GPT-2：PPL ≈ 18-35（Penn Treebank 数据集）
- GPT-3：PPL ≈ 20（One Billion Word 数据集）

**注意事项**：不同数据集上的 PPL 不可直接比较；词级别与字符级别的 PPL 也不可比较。

---

### 问题 5：不同的文本生成策略各有什么优缺点？应该如何选择？

**各生成策略对比**

| 策略 | 原理 | 优点 | 缺点 | 适用场景 |
|------|------|------|------|---------|
| **贪婪解码** | 每步选概率最大的词 | 快速、确定性 | 重复、缺乏多样性 | 简单任务、速度优先 |
| **束搜索** | 保留 k 个最优候选序列 | 质量高、避免局部最优 | 计算量大，仍可能单调 | 机器翻译、摘要 |
| **温度采样** | 调整概率分布尖锐度 | 可控多样性 | 低质量词可能被选中 | 创意写作 |
| **Top-k 采样** | 只从概率最高的 k 个词中采样 | 避免低概率词 | k 值固定，不灵活 | 通用文本生成 |
| **Top-p (Nucleus) 采样** | 从累积概率达到 p 的最小词集中采样 | 自适应候选集大小 | 需要调参 | 高质量创意生成 |

**温度参数的影响**

$$P_i^{(T)} = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- $T \to 0$：趋近贪婪解码，确定性最强
- $T = 1$：原始概率分布
- $T > 1$：分布趋于均匀，随机性增强

**实践建议**

- 需要**准确性**的任务（翻译、问答）：使用束搜索
- 需要**多样性**的任务（对话、创意写作）：使用 Top-p 采样（p=0.9）+ 适中温度（T=0.7-1.0）
- **生产环境**：Top-p + Top-k 组合使用，加入重复惩罚（repetition penalty）
- **研究/探索**：多种策略对比，根据下游评估指标选择

最佳实践：先用 Top-p=0.9 作为基线，根据生成质量调整温度参数。